# TP 2 — Solutions: First Steps in Python

**Module**: Programmation Python — ELNI 5.5  
**For instructor use / post-lab release**

---

This notebook contains full worked solutions for TP 2 with explanations and notes on alternative approaches.


---

## Solution — Exercise 1: Unit Conversion Calculator

The key insight here is naming variables with units embedded in the name (e.g., `length_mm` vs `length_m`) to make conversions self-documenting.


In [ ]:
import math

# 1. mm → m
length_mm = 250
length_m = length_mm / 1000
print(f"{length_mm} mm = {length_m:.3f} m")

# 2. kN → N
force_kn = 12.5
force_n = force_kn * 1000
print(f"{force_kn} kN = {force_n:.3f} N")

# 3. MPa → Pa
stress_mpa = 45
stress_pa = stress_mpa * 1e6
print(f"{stress_mpa} MPa = {stress_pa:.3f} Pa")

# 4. Circular column area: A = pi * r^2
diameter_mm = 400
diameter_m = diameter_mm / 1000
radius_m = diameter_m / 2
area_m2 = math.pi * radius_m ** 2
print(f"Column area (d={diameter_mm} mm) = {area_m2:.6f} m²")

# 5. Floor division and modulus
total_seconds = 1350
minutes = total_seconds // 60
seconds = total_seconds % 60
print(f"{total_seconds} s = {minutes} min {seconds} s")

**Note on** `1e6`: This is Python's scientific notation for 10⁶. It produces a `float`. `1e6 == 1000000.0`. Equivalent alternatives: `10**6` (gives `int 1000000`), `1_000_000` (underscores for readability, Python 3.6+).


---

## Solution — Exercise 2: Formatted Output

We use `zip()` to iterate over three parallel lists simultaneously. The format specifiers `:<8`, `:>12.1f`, `:>16.2f` ensure consistent column widths.


In [ ]:
timestamp = "2024-01-15 08:00:00"
sensors = ["S1", "S2", "S3"]
temperatures = [22.3, 23.1, -999.0]
pressures = [101.20, 101.50, 101.60]
statuses = ["OK", "OK", "ERROR"]

separator = "=" * 50
thin_line = "-" * 50

print(separator)
print(f"  Sensor Report — {timestamp}")
print(separator)
print(f"{'Sensor':<8} {'Temp (°C)':>12} {'Pressure (kPa)':>16} {'Status':<8}")
print(thin_line)

for sensor, temp, pressure, status in zip(sensors, temperatures, pressures, statuses):
    print(f"{sensor:<8} {temp:>12.1f} {pressure:>16.2f} {status:<8}")

print(thin_line)

**Discussion**: `zip()` is one of Python's most useful built-ins for working with parallel lists. It creates an iterator of tuples pairing corresponding elements. Here it removes the need for index-based access (`temperatures[i]`) and produces cleaner, more readable code. We will see dictionaries in Chapter 3 which often replace parallel lists entirely.


---

## Solution — Exercise 3: Variables and Types


In [ ]:
raw_count = "42"
raw_temp = "22.85"
raw_status = "OK"
raw_flag = "1"
raw_missing = ""

# int conversion
count = int(raw_count)
print(f"raw: {raw_count!r} → value: {count} (type: {type(count).__name__})")

# float conversion
temperature = float(raw_temp)
print(f"raw: {raw_temp!r} → value: {temperature} (type: {type(temperature).__name__})")

# str — no conversion needed
status = raw_status
print(f"raw: {raw_status!r} → value: {status} (type: {type(status).__name__})")

# bool via int
flag = bool(int(raw_flag))
print(f"raw: {raw_flag!r} → value: {flag} (type: {type(flag).__name__})")

# None for missing
missing = float(raw_missing) if raw_missing else None
print(f"raw: {raw_missing!r} → value: {missing} (type: {type(missing).__name__})")

**Note on the `!r` conversion flag**: In an f-string, `{value!r}` formats the value using `repr()` instead of `str()`. For strings, this adds surrounding quotes and escapes special characters, making it unambiguous that you are displaying a string value. Very useful for debugging.


---

## Solution — Exercise 4: Log File Parser


In [ ]:
line1 = "[2024-01-15 08:16:00] ERROR  Failed to read S2: value below threshold"

# 1. Extract date — slice positions 1-10 (skip the opening '[')
date = line1[1:11]
print(f"Date   : {date}")

# 2. Extract time — positions 12-19
time = line1[12:20]
print(f"Time   : {time}")

# 3 & 4. Split on whitespace
parts = line1.split()
# parts[0] = '[2024-01-15', parts[1] = '08:16:00]', parts[2] = 'ERROR'
# parts[3:] = message words
level = parts[2]
message = " ".join(parts[3:])
print(f"Level  : {level}")
print(f"Message: {message}")

# 5. Check for ERROR
if level == "ERROR":
    print(f"WARNING: Error detected in log — {message}")

# 6. Word count in message
word_count = len(message.split())
print(f"Words in message: {word_count}")

# 7. Replace S2 with SENSOR_2
updated_message = message.replace("S2", "SENSOR_2")
print(f"Updated message: {updated_message}")

**Alternative for extraction**: Using `split(']', 1)` splits only at the first `]`, cleanly separating the timestamp from the rest of the line:
```python
timestamp_part, rest = line1.split('] ', 1)
# timestamp_part = '[2024-01-15 08:16:00'
# rest = 'ERROR  Failed to read S2: value below threshold'
```
This is more robust than slicing by fixed character positions, because it works even if the timestamp format changes.


---

## Solution — Exercise 5: Load Classifier


In [ ]:
allowable_kn = 100.0
test_loads_kn = [20.0, 75.0, 95.0, 120.0, -5.0]

for applied_kn in test_loads_kn:
    ratio = applied_kn / allowable_kn

    if applied_kn < 0:
        classification = "Invalid (negative load)"
    elif ratio < 0.5:
        classification = "Under-designed"
    elif ratio < 0.8:
        classification = "Acceptable"
    elif ratio <= 1.0:
        classification = "Near capacity — review"
    else:
        classification = "OVERSTRESSED"

    print(f"Applied = {applied_kn:6.1f} kN | Ratio = {ratio:5.2f} | {classification}")

**Discussion**: We check for `applied_kn < 0` before computing the ratio, because a negative ratio would give a misleading classification. Always validate inputs before using them in conditional logic. In Chapter 5 you will learn to raise exceptions for invalid inputs.


---

## Solution — Exercise 6: Design Table Generator


In [ ]:
import math

# Part A — Reynolds number table
density = 998.0
viscosity = 1.002e-3
diameter = 0.05

print(f"{'Velocity (m/s)':<18} {'Re':>10} {'Flow regime':<15}")
print("-" * 45)

for v_tenth in range(1, 11):   # 1, 2, ..., 10
    velocity = v_tenth / 10.0  # 0.1, 0.2, ..., 1.0
    Re = (velocity * diameter * density) / viscosity

    if Re < 2300:
        regime = "Laminar"
    elif Re > 4000:
        regime = "Turbulent"
    else:
        regime = "Transitional"

    print(f"{velocity:<18.1f} {Re:>10,.0f} {regime:<15}")

In [ ]:
# Part B — while loop: compression test
FAILURE_STRESS_PA = 30e6
diameter_m = 0.150
load_step_kn = 5.0

area_m2 = math.pi * (diameter_m / 2) ** 2
load_kn = 0.0
stress_pa = 0.0

print(f"{'Load (kN)':<12} {'Stress (MPa)':<15} {'Status':<10}")
print("-" * 38)

while stress_pa <= FAILURE_STRESS_PA:
    load_kn += load_step_kn
    load_n = load_kn * 1000
    stress_pa = load_n / area_m2
    stress_mpa = stress_pa / 1e6

    if stress_pa > FAILURE_STRESS_PA:
        status = "FAILURE"
    else:
        status = "OK"

    print(f"{load_kn:<12.1f} {stress_mpa:<15.2f} {status:<10}")

print(f"\nFailure at load = {load_kn:.1f} kN (stress = {stress_pa/1e6:.2f} MPa)")

**Note on the loop structure**: The `while` loop checks the condition *before* each iteration. Since we increment the load at the start of the loop body, the first check is after the first increment. This is correct here — we want to stop when the *computed* stress exceeds the threshold.


---

## Solution — Exercise 7: Interactive Converter


In [ ]:
# Fixed test values simulating user input
raw_value = "100"
unit = "C"

value = float(raw_value)
unit = unit.upper().strip()

if unit == "C":
    temp_c = value
    temp_f = value * 9 / 5 + 32
    temp_k = value + 273.15
elif unit == "F":
    temp_c = (value - 32) * 5 / 9
    temp_f = value
    temp_k = temp_c + 273.15
elif unit == "K":
    temp_c = value - 273.15
    temp_f = temp_c * 9 / 5 + 32
    temp_k = value
else:
    print(f"Unknown unit: {unit!r}. Please enter C, F, or K.")
    temp_c = temp_f = temp_k = None

if temp_c is not None:
    print(f"\nConversion results:")
    print(f"  {value:.2f} °{unit} = {temp_c:.2f} °C = {temp_f:.2f} °F = {temp_k:.2f} K")

**Alternative input handling**: The `else` branch sets all temperatures to `None` and uses `if temp_c is not None` before printing. This is a clean pattern for handling invalid input without raising an exception (though raising `ValueError` would also be appropriate — covered in Chapter 5).


---

## Solution — Stretch Goal: Combined Pipeline


In [ ]:
raw_readings = [
    "S1:22.3:OK",
    "S2:-999.0:ERROR",
    "S3:23.1:OK",
    "S1:22.8:OK",
    "S2:24.1:OK",
    "S3:23.5:OK"
]

print(f"{'Sensor':<8} {'Temp (°C)':>10} {'Temp (°F)':>10} {'Status':<8}")
print("-" * 40)

valid_count = 0
total_temp = 0.0

for raw in raw_readings:
    sensor_id, temp_str, status = raw.split(':')
    temp_c = float(temp_str)

    # Skip faulty readings
    if status == "ERROR" or temp_c < -100:
        print(f"{sensor_id:<8} {'---':>10} {'---':>10} {status:<8}  (skipped)")
        continue

    temp_f = temp_c * 9 / 5 + 32
    print(f"{sensor_id:<8} {temp_c:>10.1f} {temp_f:>10.1f} {status:<8}")

    valid_count += 1
    total_temp += temp_c

print("-" * 40)
if valid_count > 0:
    average_temp = total_temp / valid_count
    print(f"Valid readings : {valid_count}")
    print(f"Average temp   : {average_temp:.2f} °C")

In [ ]:
# While-loop version (alternative implementation)
i = 0
valid_count = 0
total_temp = 0.0

print("\n--- while loop version ---")
print(f"{'Sensor':<8} {'Temp (°C)':>10} {'Status':<8}")
print("-" * 30)

while i < len(raw_readings):
    sensor_id, temp_str, status = raw_readings[i].split(':')
    temp_c = float(temp_str)
    i += 1

    if status == "ERROR" or temp_c < -100:
        continue

    print(f"{sensor_id:<8} {temp_c:>10.1f} {status:<8}")
    valid_count += 1
    total_temp += temp_c

print(f"Valid: {valid_count}, Average: {total_temp/valid_count:.2f} °C" if valid_count else "No valid readings.")